In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import random
import imageio_ffmpeg
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Rectangle
from typing import Literal, Callable

In [ ]:
plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

In [ ]:
EPISODES = 5000
ANIM_INTERVAL = 1000
ETA = 0.1
GAMMA = 0.99
EPSILON = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.999
STEP_LIMIT = 1000
STEP_DURATION = 0.001
ANGLE_LIMIT = np.deg2rad(12)
POSITION_LIMIT = 2.4
GRAVITY = 9.81
POLE_LEN = 0.5
POLE_MASS = 0.1
CART_MASS = 1.0
FORCE_MAG = 10
BINS = [
    np.linspace(-POSITION_LIMIT, POSITION_LIMIT, 6),
    np.linspace(-3.0, 3.0, 6),
    np.linspace(-ANGLE_LIMIT, ANGLE_LIMIT, 12),
    np.linspace(-3.5, 3.5, 12)
]

In [ ]:
def cartpole_ode(t: float, state: list[float], force_func: Callable[[float, list[float]], float], m_c: float=CART_MASS, m_p: float=POLE_MASS, l: float=POLE_LEN, g: float=GRAVITY) -> list[float]:
    """
    Calculate the next velocity and position for cart and pole

    Parameters
    ----------
    t: float
        time
    state: list[float]
        previous state
    force_func: Callable[[float, list[float]], float]
        force function
    m_c: float = CART_MASS
        cart mass
    m_p: float = POLE_MASS
        pole mass
    l: float = POLE_LEN
        pole length
    g: float = GRAVITY
        gravity

    Returns
    -------
    list[float]
        new velocity, acceleration, angular velocity, and angular acceleration
    """
    x, vel, theta, a_vel = state
    F = force_func(t, state)

    total_mass = m_c + m_p
    sin_t = np.sin(theta)
    cos_t = np.cos(theta)

    # angular acceleration (pole)
    a_accel = (
            g * sin_t + cos_t * ((-F - m_p * l * (a_vel ** 2) * sin_t) / total_mass)
        ) / (
            l * (4.0 / 3.0 - (m_p * cos_t ** 2) / total_mass)
        )

    # cart acceleration
    accel = ((-F - m_p * l * (a_vel ** 2) * sin_t) / total_mass) + (m_p * l * a_accel * cos_t) / total_mass

    return [ vel, accel, a_vel, a_accel ]

def out_of_bounds(t, y):
    x, vel, theta, a_vel = y
    if x > POSITION_LIMIT or x < -POSITION_LIMIT:
        return 0
    elif theta > ANGLE_LIMIT or theta < -ANGLE_LIMIT:
        return 0

    return 1

out_of_bounds.terminal = True

In [ ]:
def discretize_state(state: list[float]) -> tuple[int, int, int, int]:
    """
    Discretize the current state

    Parameters
    ----------
    state: list[float]
        current state

    Returns
    -------
    int
        discretized cart pos
    int
        discretized cart velocity
    int
        discretized pole angle
    int
        discretized pole ang velocity
    """
    discrete = []
    for i in range(len(state)):
        discrete.append(np.digitize(state[i], BINS[i]))

    return tuple(discrete)

In [ ]:
def animate(t: np.array[float], x_cart: np.array[float], theta: np.array[float], l: float, name: str) -> None:
    """
    Animate the series of states

    Parameters
    ----------
    t: np.array[float]
        numpy array of times
    x_cart: np.array[float]
        numpy array of cart positions
    theta: np.array[float]
        numpy array of pole angles
    l: float
        pole length
    name: str
        file name
    """
    fig, ax = plt.subplots(figsize=(6,4))
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect("equal")
    ax.grid(True)

    cart_w, cart_h = 0.4, 0.2
    cart = Rectangle((0,0), cart_w, cart_h, fc="royalblue", ec="black")
    ax.add_patch(cart)
    pole, = ax.plot([], [], lw=3, color="crimson")
    bob, = ax.plot([], [], "o", markersize=8, color="black")

    def update(frame: int) -> tuple[Rectangle, Axes, Axes]:
        """
        Update drawing with the next frame

        Parameters
        ----------
        frame: int
            frame number

        Returns
        -------
        Rectangle
            cart patch
        Axes
            pole patch
        Axes
            pole bob patch
        """
        cx = x_cart[frame]
        ang = theta[frame]

        cart.set_xy((cx - cart_w / 2, -cart_h / 2))

        px = cx + l * np.sin(ang)
        py = l * np.cos(ang)

        pole.set_data([cx, px], [0, py])
        bob.set_data([px], [py])

        return cart, pole, bob

    ani = FuncAnimation(fig, update, frames=len(t), interval=1, blit=True)
    plt.show()

    writer = FFMpegWriter(fps=1000, bitrate=1800)
    ani.save(name, writer=writer, dpi=150)

In [ ]:
class CartPoleEnv:
    def __init__(self):
        self.x = 0.0 # cart position
        self.theta = 0.05 # pole angle
        self.vel = 0.0 # cart velocity
        self.a_vel = 0.0 # pole angular velocity
        self.m_c = CART_MASS # cart mass
        self.m_p = POLE_MASS # pole mass
        self.l = POLE_LEN # pole length
        self.f_mag = FORCE_MAG # movement force magnitude
        self.tau = STEP_DURATION # time per step

        # random state
        self.state = [ random.uniform(-0.05, 0.05) for _ in range(4) ]

        # histories for animation
        self.time_hist = []
        self.x_hist = []
        self.theta_hist = []

    def reset(self) -> list[float]:
        """
        Reset the state of the environment

        Returns
        -------
        list[float]
            new state
        """
        self.state = [ random.uniform(-0.05, 0.05) for _ in range(4) ]
        self.time_hist = []
        self.x_hist = []
        self.theta_hist = []
        return self.state

    def gen_animation(self, name: str) -> None:
        """
        Animate the most recent episode run in the environment

        Parameters
        ----------
        name: str
            filename
        """
        animate(np.array(self.time_hist), np.array(self.x_hist), np.array(self.theta_hist), self.l, name)

    def step(self, action: Literal[0, 1], n: int) -> tuple[list[float], float, bool]:
        """
        Run a step

        Parameters
        ----------
        action: Literak[0, 1]
            left or right movement (0 or 1)
        n: int
            current step in the episode

        Returns
        -------
        list[float]
            new state
        float
            reward
        bool
            done flag
        """
        # apply force dependent on movement dir
        if action == 0:
            force = -FORCE_MAG
        else:
            force = FORCE_MAG

        # solve for pos, vel, theta, a_vel over time
        sol = solve_ivp(
            fun=lambda t, y: cartpole_ode(t, y, force_func=lambda t, s: force),
            t_span=(0.0, self.tau),
            y0=self.state,
            t_eval=[ 0.0, self.tau ],
            events=out_of_bounds
        )

        # set state to the final position, velocity, angle, and angular velocity
        self.state = [ sol.y[0][len(sol.y[0]) - 1], sol.y[1][len(sol.y[1]) - 1], sol.y[2][len(sol.y[2]) - 1], sol.y[3][len(sol.y[3]) - 1]]

        # update histories for animation
        self.time_hist.extend((sol.t + (self.tau * n)).tolist())
        self.x_hist.extend(sol.y[0].tolist())
        self.theta_hist.extend(sol.y[2].tolist())

        # check fail condition
        done = bool(sol.status == 1)
        reward = 1.0 if not done else 0.0

        return self.state, reward, done


In [ ]:
# define Q-table
# 2 actions (L, R) per discritized position, velocity, angle, and angular velocity
q_table_shape = tuple(len(b) + 1 for b in BINS) + (2,)
q_table = np.zeros(q_table_shape)

# editable copy of the constant
epsilon = EPSILON

# create environment
env = CartPoleEnv()

# score list for aggregation
scores = []

for ep in range(EPISODES):
    # discretize the current environment state
    state = discretize_state(env.reset())
    total_reward = 0
    done = False
    n = 0

    # step through episode
    while not done and n < STEP_LIMIT:
        n += 1

        # pick random or optimal dependent on epsilon
        if np.random.random() < epsilon:
            action = np.random.choice([ 0, 1 ])
        else:
            action = np.argmax(q_table[state])

        next_state_raw, reward, done = env.step(action, n)
        next_state = discretize_state(next_state_raw)

        best_next_q = np.max(q_table[next_state])
        td_target = reward + GAMMA * best_next_q * (not done)
        q_table[state][action] += ETA * (td_target - q_table[state][action])

        state = next_state
        total_reward += reward

    # update epsilon to increase exploitation
    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

    # append score for 100s aggregation
    scores.append(total_reward)

    # create animation on set interval
    if (ep + 1) % ANIM_INTERVAL == 0 or ep == 0:
        env.gen_animation(f"episode_{ep + 1}.mp4")

    # 100-ep aggregation
    if (ep + 1) % 100 == 0:
        avg_score = np.mean(scores[-100:])
        print(f"Episode {ep + 1}/{EPISODES} | Avg Score (past 100 eps): {avg_score} / {STEP_LIMIT} max |"
        f" Epsilon: {epsilon:.3f}")
